# Investigação do colapso do Softmax no MNIST

Notebook Python reproduzível que lê a evidência consolidada dos runs controlados e refaz as principais checagens do relatório. Ele não depende de TensorFlow, pandas ou bibliotecas de visualização: usa apenas a biblioteca padrão para continuar executável mesmo quando a aplicação HTML não carrega.

> Execute a partir da raiz do repositório ou ajuste `EVIDENCE_PATH` na célula de carregamento.

## tl;dr

O Softmax usado repetidamente nas camadas ocultas colapsou a representação: obteve 11,25% de acurácia, F1 macro de 2,02%, previu a classe 1 em 100% do teste e produziu logits quase idênticos entre imagens. ReLU e Sigmoid, sob o mesmo protocolo, ficaram perto de 99% de F1 macro.

## Context & Methods

A comparação usa MNIST, seed 42, split estratificado local 70/15/15, batch size 256, learning rate 0,0003, 100 épocas, `mixed_float16`, normalização `unit_interval`, `all_raw` e a mesma política de aumento de dados. A única variável experimental pretendida é a ativação: ReLU, Sigmoid ou Softmax.

A investigação lê `analysis_reports/investigacao_mnist_softmax.json`, que foi gerado a partir dos artefatos dos runs e do probe de ativações intermediárias. A célula final contém asserções de reconciliação para impedir que números inconsistentes passem despercebidos.

In [ ]:
from pathlib import Path
import json
import math

EVIDENCE_PATH = Path("analysis_reports/investigacao_mnist_softmax.json")
if not EVIDENCE_PATH.exists():
    candidates = list(Path.cwd().parents) + [Path.cwd()]
    EVIDENCE_PATH = next((p / "analysis_reports/investigacao_mnist_softmax.json" for p in candidates if (p / "analysis_reports/investigacao_mnist_softmax.json").exists()), EVIDENCE_PATH)

evidence = json.loads(EVIDENCE_PATH.read_text(encoding="utf-8"))
experiment = evidence["experiment"]
comparison = {row["activation"].title(): row for row in evidence["comparison"]}
print(f"Evidência: {EVIDENCE_PATH.resolve()}")
print(f"Ativações: {experiment['activations']} | teste: {experiment['test_samples']} exemplos")
print(f"Split fingerprint: {experiment['split_fingerprint']}")

## Data

A fonte primária dos números abaixo são os artefatos salvos pelo pipeline. Os caminhos de código e de saída são impressos para rastreabilidade.

In [ ]:
print("Caminhos de origem:")
for name, path in evidence["source_paths"].items():
    print(f"- {name}: {path}")

## Results

### Comparação de desempenho

In [ ]:
def pct(value):
    return f"{100 * value:.2f}%"

print(f"{'Ativação':<10} {'Acurácia':>10} {'Bal. acc.':>10} {'F1 macro':>10} {'Loss':>10} {'N classes':>10}")
print("-" * 66)
for activation in ("Relu", "Sigmoid", "Softmax"):
    row = comparison[activation]
    print(f"{activation:<10} {pct(row['test_accuracy']):>10} {pct(row['test_balanced_accuracy']):>10} {pct(row['test_macro_f1']):>10} {row['test_loss']:>10.3f} {row['predicted_class_count']:>10}")

In [ ]:
def bar(value, width=50):
    return "█" * max(1, round(value * width))

print("F1 macro no teste")
for activation in ("Relu", "Sigmoid", "Softmax"):
    value = comparison[activation]["test_macro_f1"]
    print(f"{activation:<8} {pct(value):>7} |{bar(value)}")

### Distribuição das previsões

O modelo Softmax previu somente a classe 1. A acurácia não é uma aproximação: ela é exatamente a participação da classe 1 no conjunto de teste.

In [ ]:
softmax_distribution = evidence["prediction_distribution"]["softmax"]
true_total = sum(row["true_count"] for row in softmax_distribution)
predicted_total = sum(row["predicted_count"] for row in softmax_distribution)
class_one = next(row for row in softmax_distribution if row["label"] == 1)
accuracy_by_mode = class_one["true_count"] / true_total
print(f"Total verdadeiro: {true_total}")
print(f"Total previsto: {predicted_total}")
print(f"Classe prevista pelo Softmax: 1 em {predicted_total}/{predicted_total} exemplos")
print(f"Participação verdadeira da classe 1: {class_one['true_count']}/{true_total} = {pct(accuracy_by_mode)}")
print("\nDígito | verdadeiro | previsto")
for row in softmax_distribution:
    print(f"{row['label']:>6} | {row['true_count']:>10} | {row['predicted_count']:>8}")

### Variação das ativações intermediárias

`mean_feature_std_across_samples` mede quanto cada característica varia entre 256 imagens do primeiro lote de teste. A unidade absoluta não deve ser comparada sem cuidado entre ativações diferentes; a queda progressiva dentro da cadeia Softmax e a invariância dos logits são os sinais principais.

In [ ]:
selected_layers = ["block1_activation1", "block3_activation1", "block5_activation2", "dense2", "logits"]
intermediate = {(row["activation"].title(), row["layer"]): row for row in evidence["intermediate"]}
print(f"{'Camada':<22} {'ReLU':>14} {'Sigmoid':>14} {'Softmax':>14}")
print("-" * 68)
for layer in selected_layers:
    values = []
    for activation in ("Relu", "Sigmoid", "Softmax"):
        value = intermediate[(activation, layer)]["mean_feature_std_across_samples"]
        values.append(f"{value:.6g}")
    print(f"{layer:<22} {values[0]:>14} {values[1]:>14} {values[2]:>14}")

### Interpretação arquitetural

O Keras aplica Softmax no eixo final por padrão. Em convoluções com formato channels-last, esse eixo é o canal; em Dense, são as 256 unidades. Assim, cada vetor vira uma distribuição não negativa de soma 1. A aplicação repetida após `GroupNormalization` reduz a escala e a diversidade da representação.

A saída final, porém, é linear e a loss usa `SparseCategoricalCrossentropy(from_logits=True)`. Portanto, o problema não é Softmax duplicado na saída: o problema é usar Softmax como ativação oculta em todos os blocos.

In [ ]:
softmax = comparison["Softmax"]
chance_loss = math.log(10)
print(f"Loss Softmax: {softmax['test_loss']:.6f}")
print(f"ln(10), referência de dez classes: {chance_loss:.6f}")
print(f"Melhor época de validação: {softmax['best_val_epoch']}")
print(f"F1 macro inicial/final: {pct(softmax['initial_val_macro_f1'])} / {pct(softmax['final_val_macro_f1'])}")
print(f"Maior diferença absoluta entre logits e a primeira imagem: {softmax['logit_max_abs_difference_from_first']:.6g}")

## Takeaways

1. A recomendação é manter ReLU ou Sigmoid nas camadas ocultas.
2. Softmax deve ficar na decodificação probabilística ou na saída final com `from_logits=False`.
3. Para experimentos adicionais, comparar Softmax em apenas uma camada, `axis` explícito, conexões residuais e `float32`.
4. Além de acurácia/F1, monitorar número de classes previstas, entropia da distribuição e variação dos logits.

### Checagens e fontes

As checagens abaixo reconciliam totais, denominadores e a acurácia gerada pela classe dominante. As fontes técnicas são: [Keras Softmax](https://keras.io/api/layers/activation_layers/softmax/), [Keras Dense](https://keras.io/api/layers/core_layers/dense/), [Keras GroupNormalization](https://keras.io/2/api/layers/normalization_layers/group_normalization/), [TensorFlow SparseCategoricalCrossentropy](https://www.tensorflow.org/api_docs/python/tf/keras/losses/SparseCategoricalCrossentropy) e [MNIST](https://yann.lecun.com/exdb/mnist/).

In [ ]:
assert len(comparison) == 3
assert set(comparison) == {"Relu", "Sigmoid", "Softmax"}
assert true_total == experiment["test_samples"]
assert predicted_total == experiment["test_samples"]
assert softmax["predicted_class_count"] == 1
assert softmax["predicted_mode_share"] == 1.0
assert abs(accuracy_by_mode - softmax["test_accuracy"]) < 1e-12
assert softmax["logit_max_abs_difference_from_first"] < 1e-5
print("Todas as checagens de consistência passaram.")